# 📦 [EN] Phase 3: Prescriptive Analytics (Inventory Optimization)
# 📦 [ES] Fase 3: Analítica Prescriptiva (Optimización de Inventario)

> **[EN] Business Objective:** Determine the optimal quantity of units to purchase for each product line to maximize total profit, subject to budget and warehouse space constraints.
> 
> **[ES] Objetivo de Negocio:** Determinar la cantidad óptima de unidades a comprar de cada línea de producto para maximizar la ganancia total, sujeto a restricciones de presupuesto y espacio en almacén.

---

### 🧮 [EN] Mathematical Formulation / [ES] Formulación Matemática

**[EN] Decision Variables / [ES] Variables de Decisión:**
Let $x_i$ be the number of units to purchase for product line $i$.
Sea $x_i$ el número de unidades a comprar para la línea de producto $i$.

**[EN] Objective Function (Maximize Profit) / [ES] Función Objetivo (Maximizar Ganancia):**
$$ \text{Maximize } Z = \sum_{i=1}^{n} (\text{Profit}_i \cdot x_i) $$

**[EN] Constraints / [ES] Restricciones:**
1. **Budget Constraint / Restricción de Presupuesto:** The total cost of purchased units cannot exceed the maximum budget ($B$).
$$ \sum_{i=1}^{n} (\text{Cost}_i \cdot x_i) \le B $$

2. **Space Constraint / Restricción de Espacio:** The total volume of purchased units cannot exceed the warehouse capacity ($V$).
$$ \sum_{i=1}^{n} (\text{Volume}_i \cdot x_i) \le V $$

3. **Demand Constraint / Restricción de Demanda:** We should not buy more units than the forecasted maximum demand ($D_i$).
$$ x_i \le D_i \quad \forall i $$

4. **Non-negativity / No negatividad:**
$$ x_i \ge 0 \quad \text{and integer} $$

In [2]:
import pulp
import pandas as pd
import logging

# [EN] Logger Configuration / [ES] Configuración de logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_optimization_data() -> pd.DataFrame:
    """
    [EN] Creates a DataFrame with parameters for linear programming.
    [ES] Crea un DataFrame con los parámetros para la programación lineal.
    """
    logging.info("[EN] Initializing inventory parameters... / [ES] Inicializando parámetros de inventario...")
    
    # [EN] We define realistic parameters for our 6 product lines
    # [ES] Definimos parámetros realistas para nuestras 6 líneas de producto
    data = {
        'product_line': [
            'Health and beauty', 'Electronic accessories', 'Home and lifestyle',
            'Sports and travel', 'Food and beverages', 'Fashion accessories'
        ],
        'unit_cost': [25.0, 55.0, 40.0, 35.0, 10.0, 30.0],       # Cost to buy ($)
        'unit_profit': [8.0, 15.0, 12.0, 10.0, 3.0, 9.0],        # Net profit ($)
        'unit_volume': [0.05, 0.15, 0.50, 0.30, 0.10, 0.08],     # Space needed (m^3)
        'max_demand': [500, 300, 400, 450, 1200, 600]            # Forecasted max units (from Prophet)
    }
    
    return pd.DataFrame(data)

# --- [EN] Execution Block / [ES] Bloque de Ejecución ---
df_inventory = setup_optimization_data()

# [EN] Define Business Constraints / [ES] Definir Restricciones de Negocio
MAX_BUDGET = 30000.0  # [EN] Maximum money to spend ($) / [ES] Dinero máximo a gastar ($)
MAX_SPACE = 200.0     # [EN] Maximum warehouse space (m^3) / [ES] Espacio máximo en almacén (m^3)

display(df_inventory)
logging.info(f"[EN] Budget: ${MAX_BUDGET} | Space: {MAX_SPACE}m^3 / [ES] Presupuesto: ${MAX_BUDGET} | Espacio: {MAX_SPACE}m^3")

2026-09-14 14:23:13,216 - INFO - [EN] Initializing inventory parameters... / [ES] Inicializando parámetros de inventario...


,product_line,unit_cost,unit_profit,unit_volume,max_demand
0,Health and beauty,25.0,8.0,0.05,500
1,Electronic accessories,55.0,15.0,0.15,300
2,Home and lifestyle,40.0,12.0,0.50,400
3,Sports and travel,35.0,10.0,0.30,450
4,Food and beverages,10.0,3.0,0.10,1200
5,Fashion accessories,30.0,9.0,0.08,600


2026-09-14 14:23:13,243 - INFO - [EN] Budget: $30000.0 | Space: 200.0m^3 / [ES] Presupuesto: $30000.0 | Espacio: 200.0m^3


In [3]:
import pulp
import pandas as pd
import logging

def solve_inventory_optimization(df: pd.DataFrame, max_budget: float, max_space: float):
    """
    [EN] Builds and solves the linear programming model using PuLP.
    [ES] Construye y resuelve el modelo de programación lineal usando PuLP.
    """
    logging.info("[EN] Building PuLP model... / [ES] Construyendo modelo PuLP...")
    
    # 1. [EN] Initialize Model (Goal is to MAXIMIZE) / [ES] Inicializar Modelo (El objetivo es MAXIMIZAR)
    prob = pulp.LpProblem("Supermarket_Inventory_Optimization", pulp.LpMaximize)
    
    # 2. [EN] Decision Variables (Must be integers >= 0) / [ES] Variables de Decisión (Enteros >= 0)
    products = df['product_line'].tolist()
    x = pulp.LpVariable.dicts("units", products, lowBound=0, cat='Integer')
    
    # 3. [EN] Objective Function / [ES] Función Objetivo
    prob += pulp.lpSum([df.loc[df['product_line'] == p, 'unit_profit'].values[0] * x[p] for p in products]), "Total_Profit"
    
    # 4. [EN] Constraints / [ES] Restricciones
    # a) Presupuesto
    prob += pulp.lpSum([df.loc[df['product_line'] == p, 'unit_cost'].values[0] * x[p] for p in products]) <= max_budget, "Max_Budget"
    
    # b) Espacio
    prob += pulp.lpSum([df.loc[df['product_line'] == p, 'unit_volume'].values[0] * x[p] for p in products]) <= max_space, "Max_Space"
    
    # c) Demanda Máxima (No comprar más de lo que Prophet dice que venderemos)
    for p in products:
        max_d = df.loc[df['product_line'] == p, 'max_demand'].values[0]
        prob += x[p] <= max_d, f"Max_Demand_{p.replace(' ', '_')}"
        
    # 5. [EN] Solve / [ES] Resolver
    logging.info("[EN] Solving linear programming problem... / [ES] Resolviendo problema de programación lineal...")
    prob.solve()
    
    # 6. [EN] Extract Results / [ES] Extraer Resultados
    status = pulp.LpStatus[prob.status]
    logging.info(f"[EN] Model Status: {status} / [ES] Estado del Modelo: {status}")
    
    if status == 'Optimal':
        results = []
        for p in products:
            results.append({
                'product_line': p,
                'optimal_units_to_buy': x[p].varValue
            })
        
        df_results = pd.DataFrame(results)
        
        # [EN] Merge to calculate final metrics / [ES] Unir para calcular métricas finales
        df_final = pd.merge(df, df_results, on='product_line')
        df_final['total_cost'] = df_final['unit_cost'] * df_final['optimal_units_to_buy']
        df_final['total_volume'] = df_final['unit_volume'] * df_final['optimal_units_to_buy']
        df_final['projected_profit'] = df_final['unit_profit'] * df_final['optimal_units_to_buy']
        
        return prob, df_final
    else:
        logging.error("[EN] Optimal solution not found. / [ES] Solución óptima no encontrada.")
        return prob, None

# --- [EN] Execution Block / [ES] Bloque de Ejecución ---
prob, df_optimal = solve_inventory_optimization(df_inventory, MAX_BUDGET, MAX_SPACE)

if df_optimal is not None:
    display(df_optimal[['product_line', 'optimal_units_to_buy', 'total_cost', 'total_volume', 'projected_profit']])
    
    print("\n" + "=" * 55)
    print(f"🏆 [EN] MAXIMUM PROJECTED PROFIT / [ES] GANANCIA MÁXIMA: ${df_optimal['projected_profit'].sum():,.2f}")
    print("=" * 55)
    print(f"💰 [EN] Budget Used / [ES] Presupuesto Usado: ${df_optimal['total_cost'].sum():,.2f} / ${MAX_BUDGET:,.2f}")
    print(f"📦 [EN] Space Used / [ES] Espacio Usado: {df_optimal['total_volume'].sum():,.2f} / {MAX_SPACE:,.2f} m^3")
    print("=" * 55)

2026-09-14 14:23:13,270 - INFO - [EN] Building PuLP model... / [ES] Construyendo modelo PuLP...
2026-09-14 14:23:13,309 - INFO - [EN] Solving linear programming problem... / [ES] Resolviendo problema de programación lineal...
2026-09-14 14:23:13,343 - INFO - [EN] Model Status: Optimal / [ES] Estado del Modelo: Optimal


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/pablosantana/anaconda3/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/b66b04078f94445b98b24b3f38a3aaac-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/b66b04078f94445b98b24b3f38a3aaac-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 13 COLUMNS
At line 50 RHS
At line 59 BOUNDS
At line 66 ENDATA
Problem MODEL has 8 rows, 6 columns and 18 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 9250 - 0.00 seconds
Cgl0004I processed model has 2 rows, 6 columns (6 integer (0 of which binary)) and 12 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -9238 found by DiveCoefficient after 0 iterations and 0 nodes (0.01 seconds)
Cbc0038I Full problem 2 rows 6 columns, reduced to 2 rows 2 columns
Cbc0012I Integer s

,product_line,optimal_units_to_buy,total_cost,total_volume,projected_profit
0,Health and beauty,500.0,12500.0,25.0,4000.0
1,Electronic accessories,0.0,0.0,0.0,0.0
2,Home and lifestyle,100.0,4000.0,50.0,1200.0
3,Sports and travel,0.0,0.0,0.0,0.0
4,Food and beverages,1200.0,12000.0,120.0,3600.0
5,Fashion accessories,50.0,1500.0,4.0,450.0



🏆 [EN] MAXIMUM PROJECTED PROFIT / [ES] GANANCIA MÁXIMA: $9,250.00
💰 [EN] Budget Used / [ES] Presupuesto Usado: $30,000.00 / $30,000.00
📦 [EN] Space Used / [ES] Espacio Usado: 199.00 / 200.00 m^3


---

## 👤 Applied Data Scientist & Researcher / Científico de Datos Aplicado e Investigador

**Pablo Alberto Santana Flores**  
*Chemical Engineer | PhD in Marine Sciences | Decision Intelligence*  

> **🇬🇧 EN:** Thank you for exploring this phase of the **Dynamic Sales Prediction Engine**. This notebook showcases the *Descriptive* and *Predictive* foundations of our end-to-end analytical pipeline (ETL, EDA, and Meta Prophet forecasting). Building upon these insights, the project culminates in **Phase 3: Prescriptive Analytics**, where we utilize linear programming (`PuLP`) to translate sales forecasts into optimal inventory purchasing decisions to maximize profit. Feel free to connect on LinkedIn to discuss the technical architecture.  
> 
> **🇲🇽 ES:** Gracias por explorar esta fase del **Motor Dinámico de Predicción de Ventas**. Este notebook exhibe las bases *Descriptivas* y *Predictivas* de nuestro pipeline analítico de extremo a extremo (ETL, EDA y pronóstico con Meta Prophet). Partiendo de estos *insights*, el proyecto culmina en la **Fase 3: Analítica Prescriptiva**, donde utilizamos programación lineal (`PuLP`) para traducir los pronósticos de ventas en decisiones óptimas de compra de inventario para maximizar ganancias. Siéntete libre de conectar en LinkedIn para discutir la arquitectura técnica.

*   **LinkedIn:** [linkedin.com/in/pablo-santana-mx](https://mx.linkedin.com/in/pablo-santana-mx)
*   **GitHub:** [github.com/Pablo-Santana-MX](https://github.com/Pablo-Santana-MX)